In [ ]:
import os
import cv2
import dlib
import numpy as np
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Скачивание весов экстрактора лендмарков 

In [ ]:
#!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
#!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

# Извлечение лендмарок, левого и правого глаза (кадры), сохранение извлеченных данных

In [2]:
class EyeRegionExtractor:
    """Извлечение области глаз с использованием facial landmarks"""
    
    def __init__(self, predictor_path):
        """
            predictor_path: путь к предобученной модели dlib
        """
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor(predictor_path)
        
        # Индексы landmarks для глаз 
        self.left_eye_indices = [36, 37, 38, 39, 40, 41]
        self.right_eye_indices = [42, 43, 44, 45, 46, 47]
        
    def extract_eye_regions(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray)
        
        if len(faces) == 0:
            return None

        face = faces[0]
        landmarks = self.predictor(gray, face)
        landmarks_points = np.array([[p.x, p.y] for p in landmarks.parts()])

        left_eye_points = landmarks_points[self.left_eye_indices]
        left_eye_region = self._crop_eye_region(frame, left_eye_points)

        right_eye_points = landmarks_points[self.right_eye_indices]
        right_eye_region = self._crop_eye_region(frame, right_eye_points)
        
        return {
            'left_eye': left_eye_region,
            'right_eye': right_eye_region,
            'left_eye_landmarks': left_eye_points.tolist(),
            'right_eye_landmarks': right_eye_points.tolist(),
            'all_landmarks': landmarks_points.tolist()
        }
    
    def _crop_eye_region(self, frame, eye_points):
        x_min, y_min = np.min(eye_points, axis=0)
        x_max, y_max = np.max(eye_points, axis=0)

        padding_x = int((x_max - x_min) * 0.5)
        padding_y = int((y_max - y_min) * 0.5)
        
        x_min = max(0, x_min - padding_x)
        y_min = max(0, y_min - padding_y)
        x_max = min(frame.shape[1], x_max + padding_x)
        y_max = min(frame.shape[0], y_max + padding_y)
        

        if eye_region.size > 0:
            eye_region = cv2.resize(eye_region, (128, 128))
        else:
            eye_region = np.zeros((128, 128, 3), dtype=np.uint8)
        
        return eye_region


def extract_frames_features(video_folder_path, extractor, output_dir, max_frames=64):
    video_name = Path(video_folder_path).name
    video_output_dir = Path(output_dir) / video_name
    video_output_dir.mkdir(parents=True, exist_ok=True)

    frame_files = sorted(list(Path(video_folder_path).glob("frame_*.jpg")))
    frame_files = frame_files[:max_frames]
    
    features = {
        'video_name': video_name,
        'frames': [],
        'total_frames_processed': len(frame_files)
    }
    
    processed_count = 0
    for frame_path in tqdm(frame_files, desc=f"Processing {video_name}", leave=False):
        frame = cv2.imread(str(frame_path))
        
        if frame is None:
            print(f"Warning: Could not read {frame_path}")
            continue
        
        eye_data = extractor.extract_eye_regions(frame)
        
        if eye_data is not None:
            frame_index = processed_count
 
            left_eye_path = video_output_dir / f"frame_{frame_index:04d}_left.jpg"
            right_eye_path = video_output_dir / f"frame_{frame_index:04d}_right.jpg"
            
            cv2.imwrite(str(left_eye_path), eye_data['left_eye'])
            cv2.imwrite(str(right_eye_path), eye_data['right_eye'])

            frame_info = {
                'frame_index': frame_index,
                'original_frame': frame_path.name,
                'left_eye_landmarks': eye_data['left_eye_landmarks'],
                'right_eye_landmarks': eye_data['right_eye_landmarks'],
                'left_eye_path': str(left_eye_path),
                'right_eye_path': str(right_eye_path)
            }
            
            features['frames'].append(frame_info)
            processed_count += 1
        else:
            print(f"Warning: No face detected in {frame_path.name}")

    metadata_path = video_output_dir / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(features, f, indent=2)
    
    return features


def process_dataset(frames_root_dir, output_dir, predictor_path):
    
    extractor = EyeRegionExtractor(predictor_path=predictor_path)
    
    # Находим все папки с кадрами
    video_folders = [f for f in Path(frames_root_dir).iterdir() if f.is_dir()]
    
    print(f"Found {len(video_folders)} video folders")
    
    all_features = []
    failed_videos = []
    
    for video_folder in tqdm(video_folders, desc="Processing videos"):
        try:
            features = extract_frames_features(
                video_folder_path=str(video_folder),
                extractor=extractor,
                output_dir=output_dir
            )
            
            if features['frames']:
                all_features.append(features)
            else:
                failed_videos.append(video_folder.name)
                
        except Exception as e:
            print(f"Error processing {video_folder.name}: {e}")
            failed_videos.append(video_folder.name)
            
    summary_path = Path(output_dir) / "all_features.json"
    with open(summary_path, 'w') as f:
        json.dump(all_features, f, indent=2)

    if failed_videos:
        failed_path = Path(output_dir) / "failed_videos.txt"
        with open(failed_path, 'w') as f:
            for video_name in failed_videos:
                f.write(f"{video_name}\n")
                
    print(f"Processing complete")
    print(f"Successfully processed: {len(all_features)} videos")
    print(f"Failed: {len(failed_videos)} videos")
    print(f"Features saved to: {output_dir}")
    
    return all_features


In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/hweidatasetfaces/hweiDatasetFaces"  
output_dir = "/kaggle/working/extracted_eye_features/hwei" 
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat" 
    
all_features_hwei = process_dataset(
    frames_root_dir=frames_root_dir,
    output_dir=output_dir,
    predictor_path=predictor_path
    )

In [5]:
import os
import shutil
from IPython.display import FileLink

folder_path = '/kaggle/working/extracted_eye_features/hwei'
archive_path = '/kaggle/working/hwei_eyes'

shutil.make_archive(archive_path, 'zip', folder_path)

print(f"Архив создан: {archive_path}.zip")
display(FileLink(f"{archive_path}.zip"))

Архив создан: /kaggle/working/hwei_eyes.zip


/kaggle/working/hwei_eyes.zip

In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/testframesfacesv2/fakeFramesFacesV2"  
output_dir = "/kaggle/working/extracted_eye_features/test_fake" 
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat"  
    
all_features = process_dataset(
    frames_root_dir=frames_root_dir,
    output_dir=output_dir,
    predictor_path=predictor_path
    )

In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/testframesfacesv2/realFramesFacesV2"  
output_dir = "/kaggle/working/extracted_eye_features/test_real"  
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat"  
    
all_features = process_dataset(
    frames_root_dir=frames_root_dir,
    output_dir=output_dir,
    predictor_path=predictor_path
    )

In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/valframesfacesv2/fakeFramesFacesV2" 
output_dir = "/kaggle/working/extracted_eye_features/val_fake"  
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat" 
    
all_features = process_dataset(
    frames_root_dir=frames_root_dir,
    output_dir=output_dir,
    predictor_path=predictor_path
    )

In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/valframesfacesv2/realFramesFacesV2" 
output_dir = "/kaggle/working/extracted_eye_features/val_real" 
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat"  
    
all_features_val_real = process_dataset(
    frames_root_dir=frames_root_dir,
    output_dir=output_dir,
    predictor_path=predictor_path
    ) 

In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/trainframesfacesv2/fakeFramesFacesV2"  
output_dir = "/kaggle/working/extracted_eye_features/train_fake"  
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat"  
    
all_features_train_fake = process_dataset( 
    frames_root_dir=frames_root_dir,  
    output_dir=output_dir,
    predictor_path=predictor_path
    )

In [ ]:
frames_root_dir = "/kaggle/input/datasets/mariaspasyuk/trainframesfacesv2/realFramesFacesV2"  
output_dir = "/kaggle/working/extracted_eye_features/train_real"  
predictor_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat"  
      
all_features_train_real = process_dataset(
    frames_root_dir=frames_root_dir,
    output_dir=output_dir,
    predictor_path=predictor_path
    )